# Independent Misconception-Alignment Evaluator

Measures the project's core construct directly: **does a generated distractor encode the misconception the question's teacher-written distractors encode?**

## Design

Bi-encoder ranker over misconception descriptions. Query = *question + correct answer + incorrect answer chosen*; documents = all **2,587** misconception descriptions; the measure is the rank of the question's gold misconception.

A ranker rather than an N-way classifier because **27.6% of test gold misconceptions never occur in training** — a softmax over training classes would score those questions as failures regardless of distractor quality.

## Independence (enforced in code, aborts on failure)

| Requirement | How it is enforced |
|---|---|
| Not the Stage 1 retriever | Different base encoder (**mpnet** vs MiniLM); asserted |
| No label leakage | No misconception description may appear in any query; asserted |
| No generator self-report | The model's stated misconception is never an input (by construction) |
| No question contamination | Training questions disjoint from application questions; asserted |
| Validated before use | Standalone performance + positive control gate the application step |

## Splits (reuses existing question-disjoint splits)

| Role | Data |
|---|---|
| Evaluator training | train QDPs (3,507) |
| Early stopping | val QDPs (432) |
| Standalone reference | test **gold** QDPs (431) — never seen |
| Application | generated candidates for the 187 test questions |

**Expect a null.** The gated composite found +0.015 (p=0.088) and the LLM judge +0.014 (p=1.00). This instrument ranks over 2,587 classes and is *noisier*, so it has less power, not more. Its value is an independent measurement of the actual construct, not a resolution of the Stage 2 question.

Set `REPO_URL`, Runtime → GPU, Run all. ~30–45 min.

In [ ]:
# --- The ONLY cell you edit ---
REPO_URL = "https://github.com/YOUR_USERNAME/distractor.git"  # <-- set this
BRANCH   = "experiment-2-checkpoint-analysis"
BASE_MODEL = "sentence-transformers/all-mpnet-base-v2"   # must differ from Stage 1
EPOCHS = 3
SEED   = 42

In [ ]:
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > GPU"
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
import os
if not os.path.exists("distractor"):
    !git clone -b {BRANCH} {REPO_URL} distractor
%cd distractor
%pip install -q -r requirements_gpu.txt
if not os.path.exists("outputs/results/train_qdp.csv"):
    !python scripts/01_prepare_dataset.py
assert os.path.exists("datasets/misconception_mapping.csv")
print("ready")

In [ ]:
# --- Upload Stage 2 generations (exp21 G1/G4/G5; optionally stage2_generations.jsonl
#     and reranking/selections_k1.csv for the Stage 3 comparison) ---
# Accepts a zip or raw .jsonl/.csv files.
import os, glob, shutil, zipfile
os.makedirs("outputs/generation/exp21", exist_ok=True)
os.makedirs("outputs/reranking", exist_ok=True)
if not glob.glob("outputs/generation/exp21/G*_seed*.jsonl"):
    from google.colab import files
    up = files.upload()
    for name in up:
        if zipfile.is_zipfile(name):
            with zipfile.ZipFile(name) as zf: zf.extractall(".")
            os.remove(name)
for f in glob.glob("*.jsonl"):
    dest = "outputs/generation" if f.startswith("stage2_") else "outputs/generation/exp21"
    shutil.move(f, os.path.join(dest, os.path.basename(f)))
for f in glob.glob("selections_k1.csv"):
    shutil.move(f, "outputs/reranking/selections_k1.csv")
print("exp21:", sorted(os.listdir("outputs/generation/exp21")))

## Step 1 — Train the evaluator

Contamination checks run first and abort the script on failure. ~15–20 min on a T4.

In [ ]:
!python scripts/17_train_misconception_evaluator.py \
  --base-model {BASE_MODEL} --epochs {EPOCHS} --seed {SEED}

## Step 2 — Validate, then apply

The script measures the evaluator on 431 held-out **gold** distractors and runs the positive control **before** scoring any generated output. If it fails the gate it exits without producing alignment numbers — the same discipline that caught exact match and the pairwise judge.

In [ ]:
!python scripts/18_evaluate_misconception_alignment.py --seed {SEED}

In [ ]:
print(open("outputs/misconception_evaluator/report.md").read())

## Interpretation rules

**Read the gate first.** If the evaluator fails (standalone MRR ≤ the 0.1472 zero-shot floor, or C1/C2 AUC < 0.70), no alignment number below is usable. Report the failure; do not use `--force`.

**Then read the anchor before the arms.** GOLD is teacher-written distractors scored by the same evaluator on the same questions. Generated alignment should be read *relative* to it — "generated reaches X% of teacher-written alignment" — not as an absolute score, since absolute MRR over 2,587 classes is low by construction.

**The three levels answer different questions:**

| Measure | Question |
|---|---|
| `mean_rr` (primary) | Is the *typical* generated distractor misconception-aligned? |
| `best_rr` (secondary) | Does the pool contain *at least one* aligned candidate? |
| Stage 3 selected | Does selection *improve* alignment over the pool average? |

A large `best_rr` − `mean_rr` gap means aligned candidates exist but are not reliably chosen — the same headroom Stage 3 measured on gated quality, now on the construct itself.

**On the G4 vs G5 primary comparison:** significance would be a genuine surprise. Two validated instruments already found small non-significant effects, and this one is noisier. A null here is the expected outcome and constitutes a *third independent confirmation* that the effect is small — which is a stronger claim than any single instrument supports. Do not report a null from this instrument as evidence that retrieval is useless; report it as a bounded effect.

**C3 (near-miss) is a probe, not a gate.** If it lands near 0.56, the evaluator shares the limitation of every other instrument tested: it cannot separate a valid distractor from a digit-perturbed one, because that requires procedural reasoning rather than textual association. That result is itself publishable — it would mean the intervention the paper recommends was built and the constraint held.

In [ ]:
# --- Package and download (model excluded; it is large and re-trainable) ---
!cd outputs/misconception_evaluator && zip -qr /content/distractor/evaluator_results.zip . -x "model/*" "checkpoints/*"
from google.colab import files
files.download("evaluator_results.zip")